Change the corresponding parameters in the config json files

In [1]:
import json
import os

root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL_baseline_3_run"

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith(".json"):
            with open(os.path.join(root, file), "r") as f:
                config = json.load(f)
            config["seed"] = [1994, 1995, 1996]
            with open(os.path.join(root, file), "w") as f:
                json.dump(config, f, indent=4)

Check whether the config files are valid or not

In [2]:
import json
import os

# root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL_baseline_3_run"
# root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL"
root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL"

dataset_cls_map = {
    "core50": 5,
    "imageclef": 2,
    "office31": 4,
    "officecaltech": 2,
    "officehome": 13,
    "digitsdg": 2,
    "digitsfive": 2,
    "core50": 5,
    "minidomainnet": 13,
    "domainnet": 35
}

total_json_files = 0
valid_data_json_files = 0
cil_json_files = 0
dgil_json_files = 0

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith(".json"):
            total_json_files += 1
            with open(os.path.join(root, file), "r") as f:
                config = json.load(f)

            data_checked = False
            for key in dataset_cls_map:
                if config["dataset"] == key == os.path.basename(root):
                    valid_data_json_files += 1
                    assert config["init_cls"] == config["increment"] == dataset_cls_map[key], f"file: {root} {file}, dataset: {config['dataset']}, init_cls: {config['init_cls']}, increment: {config['increment']}, expected: {dataset_cls_map[key]}"
                    data_checked = True
                    break
            assert data_checked, f"file: {root} {file}, dataset: {config['dataset']}, init_cls: {config['init_cls']}, increment: {config['increment']}"


            dgil_checked = False
            if "dgil" in file:
                dgil_json_files += 1
                assert config["enable_dgil"] == True, f"file: {root} {file}, enable_dgil: {config['enable_dgil']}"
                assert config["random_reference"] == True, f"file: {root} {file}, random_reference: {config['random_reference']}"
                assert config["reference_domain_id"] == 0, f"file: {root} {file}, reference_domain_id: {config['reference_domain_id']}"
                assert config["multi_domain_base_task"] == False, f"file: {root} {file}, multi_domain_base_task: {config['multi_domain_base_task']}"
            else:
                cil_json_files += 1
                dgil_checked = True

            assert config["print_forget"] == True


print(f"total_json_files: {total_json_files}, valid_data_json_files: {valid_data_json_files}, cil_json_files: {cil_json_files}, dgil_json_files: {dgil_json_files}")

total_json_files: 158, valid_data_json_files: 158, cil_json_files: 67, dgil_json_files: 91


Generate configs based on template config

In [ ]:
import json
import os
from pprint import pprint

root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome"
# template_cil_file = os.path.join(root_folder, "dot.json")
template_dgil_file = os.path.join(root_folder, "slca_dgil.json")

# with open(template_cil_file, "r") as f:
#     cil_config = json.load(f)

with open(template_dgil_file, "r") as f:
    dgil_config = json.load(f)

for weight in [0.01, 0.1, 1, 10]:
    post_fix = f"mmda_{weight:.2f}"
    # new_cil_file = os.path.join(root_folder, f"dot_{post_fix}.json")
    new_dgil_file = os.path.join(root_folder, f"slca_dgil_{post_fix}.json")

    # print(f"Generating new_cil_file: {new_cil_file}")
    print(f"Generating new_dgil_file: {new_dgil_file}")

    # new_cil_config = cil_config.copy()
    # new_cil_config["prefix"] = f"CIL_{post_fix}"
    # new_cil_config["use_dom_con_loss"] = True
    # new_cil_config["dom_con_weight"] = weight

    new_dgil_config = dgil_config.copy()
    new_dgil_config["prefix"] = f"DGIL_{post_fix}"
    new_dgil_config["mmda_loss_weight"] = weight

    # with open(new_cil_file, "w") as f:
    #     json.dump(new_cil_config, f, indent=4)

    with open(new_dgil_file, "w") as f:
        json.dump(new_dgil_config, f, indent=4)

Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_0.01.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_0.10.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_1.00.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_10.00.json


Generate variantions of config files for different experiments.

In [6]:
import json
import os
from pprint import pprint

root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome"

hp_config_filenames = []
hp_config_paths=[]
for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith(".json"):
            for pool_size in [10, 20, 30, 40, 50]:
                if f"p{pool_size}" in file:
                    hp_config_filenames.append(file)
                    hp_config_paths.append(os.path.join(root, file))


for hp_path, hp_filename in zip(hp_config_paths, hp_config_filenames):
    with open(hp_path, "r") as f:
        hp_config = json.load(f)
        hp_prefix = hp_config["prefix"]
    for top_k in [1, 2, 3, 4, 5, 6]:
        new_hp_config = hp_config.copy()
        new_hp_config["top_k"] = top_k
        new_hp_config["prefix"] = f"{hp_prefix}_top{top_k}"

        new_hp_config_path = os.path.join(os.path.dirname(hp_path), f"{os.path.splitext(hp_filename)[0]}_top{top_k}.json")
        with open(new_hp_config_path, "w") as f:
            json.dump(new_hp_config, f, indent=4)
        print(f"New config saved to {new_hp_config_path}")


New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top1.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top2.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top3.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top4.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top5.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p30_top6.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p20_top1.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p20_top2.json
New config saved to /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/l2p_dgil_p20_top3.json
New config saved to /gpfs-flash/junlab/liyuan/

In [5]:
import json
import os
from pprint import pprint

root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome"

hp_config_filenames = []
hp_config_paths=[]
for root, dirs, files in os.walk(root_folder):
    for file in files:
        # if there exists two "top" in file, delete it
        if "top" in file:
            os.remove(os.path.join(root, file))
